# CBD Robustness Analysis**EMS Optimization Project – Gap 1 Resolution**This notebook analyzes CBD-specific simulation results to assess policy robustness under CBD-focused conditions.

### Auto-generate missing data
The cell below checks if required processed data exists and generates it automatically if missing.
This ensures each notebook can run independently from a clean state.

In [ ]:
# Auto-generate missing processed data if needed
import sys, os
from pathlib import Path

# Detect project root (works from notebooks/ directory)
_PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if not (_PROJECT_ROOT / 'scripts' / 'generate_all_data.py').exists():
    _PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(_PROJECT_ROOT))
sys.path.insert(0, str(_PROJECT_ROOT / 'src'))

from scripts.generate_all_data import ensure_data
ensure_data(_PROJECT_ROOT)

In [ ]:
import syssys.path.insert(0, '../src')import pandas as pdimport numpy as npimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport seaborn as snsplt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'font.size': 11})POLICY_COLORS = {'P0': '#e74c3c', 'P1': '#f39c12', 'P2': '#27ae60', 'CBD_ONLY': '#3498db', 'MIXED': '#9b59b6'}

## 1. Load CBD Experiment Results

In [ ]:
df = pd.read_csv('../results/analysis/simulation/cbd_experiment/cbd_experiment_results.csv')print(f"Total runs: {len(df)}")print(f"Scenario types: {df['scenario_type'].unique()}")print(f"Policies: {df['policy'].unique()}")df.head()

## 2. CBD vs Non-CBD Response Time Comparison

In [ ]:
# Compare CBD vs non-CBD metrics across policies and scenariosbaseline = df[df['scenario_type'] == 'baseline']summary = baseline.groupby('policy').agg({    'mean_response_time': ['mean', 'std'],    'coverage_8min': ['mean', 'std'],    'cbd_mean_rt': 'mean',    'cbd_coverage_8min': 'mean',    'non_cbd_mean_rt': 'mean',    'non_cbd_coverage_8min': 'mean',}).round(4)print("Baseline CBD vs Non-CBD Comparison:")print(summary.to_string())

In [ ]:
# CBD Response Time Comparison Plotfig, axes = plt.subplots(1, 2, figsize=(14, 6))# Response time comparisonpolicies = ['P0', 'P1', 'P2']x = np.arange(len(policies))width = 0.25for ax, (overall_col, cbd_col, noncbd_col, title) in zip(axes, [    ('mean_response_time', 'cbd_mean_rt', 'non_cbd_mean_rt', 'Mean Response Time (min)'),    ('coverage_8min', 'cbd_coverage_8min', 'non_cbd_coverage_8min', '8-Minute Coverage'),]):    for i, label, col in zip(range(3), ['Overall', 'CBD', 'Non-CBD'], [overall_col, cbd_col, noncbd_col]):        vals = [baseline[baseline['policy']==p][col].mean() for p in policies]        ax.bar(x + (i-1)*width, vals, width, label=label, alpha=0.8)    ax.set_xticks(x)    ax.set_xticklabels(['P0\n(Uniform)', 'P1\n(Proportional)', 'P2\n(Optimized)'])    ax.set_title(title, fontweight='bold')    ax.set_ylabel(title)    ax.legend()    ax.grid(axis='y', alpha=0.3)fig.suptitle('CBD vs Non-CBD Performance Under Baseline Conditions', fontweight='bold', fontsize=14)plt.tight_layout()plt.savefig('../results/analysis/figures/cbd_response_comparison.png')
plt.show()print("Saved cbd_response_comparison.png")

## 3. CBD Scenario Analysis

In [ ]:
# Compare scenariosfig, axes = plt.subplots(2, 2, figsize=(14, 10))scenario_types = ['baseline', 'cbd_surge', 'cbd_slow_service']policies_main = ['P0', 'P1', 'P2']for ax, (metric, title) in zip(axes.flat, [    ('mean_response_time', 'Mean Response Time (min)'),    ('coverage_8min', '8-Minute Coverage'),    ('cbd_mean_rt', 'CBD Response Time (min)'),    ('cbd_coverage_8min', 'CBD 8-min Coverage'),]):    x = np.arange(len(scenario_types))    width = 0.25    for i, policy in enumerate(policies_main):        vals = []        for st in scenario_types:            mask = (df['scenario_type']==st) & (df['policy']==policy)            vals.append(df[mask][metric].mean())        ax.bar(x + (i-1)*width, vals, width, label=policy,               color=POLICY_COLORS.get(policy, 'gray'), alpha=0.8)    ax.set_xticks(x)    ax.set_xticklabels(['Baseline', 'CBD Surge\n(2x demand)', 'CBD Slow\nService'], fontsize=9)    ax.set_title(title, fontweight='bold')    ax.legend()    ax.grid(axis='y', alpha=0.3)fig.suptitle('Policy Performance Across CBD Scenarios', fontweight='bold', fontsize=14)plt.tight_layout()plt.savefig('../results/analysis/figures/cbd_scenario_comparison.png')
plt.show()print("Saved cbd_scenario_comparison.png")

## 4. CBD Coverage Spatial Analysis

In [ ]:
# CBD coverage heatmap across scenarios and policiesall_scenarios = df['scenario_type'].unique()all_policies = df['policy'].unique()pivot_rt = df.groupby(['scenario_type', 'policy'])['cbd_mean_rt'].mean().unstack()pivot_cov = df.groupby(['scenario_type', 'policy'])['cbd_coverage_8min'].mean().unstack()fig, axes = plt.subplots(1, 2, figsize=(14, 5))sns.heatmap(pivot_rt, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=axes[0], linewidths=0.5)axes[0].set_title('CBD Mean Response Time (min)', fontweight='bold')sns.heatmap(pivot_cov, annot=True, fmt='.3f', cmap='RdYlGn', ax=axes[1], linewidths=0.5)axes[1].set_title('CBD 8-min Coverage', fontweight='bold')fig.suptitle('CBD Performance Heatmap', fontweight='bold', fontsize=14)plt.tight_layout()plt.savefig('../results/analysis/figures/cbd_heatmap.png')
plt.show()print("Saved cbd_heatmap.png")

## 5. Summary Tables

In [ ]:
# Generate summary tablessummary_all = df.groupby(['scenario_type', 'policy']).agg({    'mean_response_time': 'mean',    'coverage_8min': 'mean',    'cbd_mean_rt': 'mean',    'cbd_coverage_8min': 'mean',    'non_cbd_mean_rt': 'mean',    'non_cbd_coverage_8min': 'mean',    'queue_fraction': 'mean',    'total_incidents': 'mean',}).round(4)summary_all.to_csv('../results/analysis/tables/cbd_summary_all.csv')
print("CBD Summary Table:")print(summary_all.to_string())# CBD-specific improvement tablebaseline_p0 = df[(df['scenario_type']=='baseline') & (df['policy']=='P0')]baseline_p2 = df[(df['scenario_type']=='baseline') & (df['policy']=='P2')]print(f"\nP2 vs P0 CBD improvement:")print(f"  CBD RT reduction: {baseline_p0['cbd_mean_rt'].mean():.2f} → {baseline_p2['cbd_mean_rt'].mean():.2f} min")print(f"  CBD Coverage gain: {baseline_p0['cbd_coverage_8min'].mean():.3f} → {baseline_p2['cbd_coverage_8min'].mean():.3f}")

## 6. Key Findings1. **P2 dominates in CBD**: The optimized policy (P2) achieves the best CBD response times across all scenarios2. **CBD surge resilience**: Even under 2× CBD demand, P2 maintains high coverage3. **Service time sensitivity**: CBD-specific service time increases have minimal impact on P2's advantage4. **Queue behavior**: No queuing observed under any CBD scenario with K=20 units